# AMAC Multimodal Foundation Model Fine-Tuning (Prithvi-EO-2.0 via TerraTorch)

**Companion notebook to `AMAC_GEE_baseline.js` and the manuscript's Section 4.3 / Section 7.**

This notebook fine-tunes a pretrained geospatial foundation model (Prithvi-EO-2.0, via IBM/NASA's [TerraTorch](https://github.com/IBM/terratorch) framework) on the same AMAC GEDI-referenced AGB task as the classical Random Forest baseline reported in the manuscript's Section 6, so the two can be compared directly (manuscript Section 7, comparison (i)).

### Before running this notebook
1. Run `AMAC_GEE_baseline.js` in the [Earth Engine Code Editor](https://code.earthengine.google.com) (with the exports section included) and let the three export tasks finish in the **Tasks** tab:
   - `AMAC_S2_HLS6band_stack` (GeoTIFF) — the 6-band optical stack, remapped to Prithvi's expected Blue/Green/Red/NIR-narrow/SWIR1/SWIR2 band order.
   - `AMAC_GEDI_points_lonlat_agbd` (CSV) — all 10,978 GEDI points with `lon`, `lat`, `agbd` columns.
   - `AMAC_AGC_baseline_RF_MgCha` (GeoTIFF) — the Section 6 RF baseline map, useful for a visual side-by-side later, not required for training.
2. These land in a folder named `AMAC_carbon` in your Google Drive.
3. Run this notebook in **Google Colab with a GPU runtime** (Runtime → Change runtime type → GPU). CPU will technically work but fine-tuning a ViT backbone will be very slow.

### A note on reliability
Getting the GEE script right took several rounds of real debugging (date-range coverage, sparse-data sampling, mask mismatches). This notebook is very likely to need similar iteration — TerraTorch's exact registry names and API surface move between versions, and I can't execute this code myself to verify it end-to-end before you run it. Cells that are most likely to need adjustment are flagged with **⚠️ FRAGILE** in their markdown header. Run cells one at a time and share the actual error text back if something breaks — that's exactly the debugging loop that got the GEE script working.

## 1. Install dependencies

In [ ]:
!pip install -q terratorch[peft] rasterio geopandas scikit-learn pandas numpy matplotlib

## 2. Mount Google Drive and locate the exported files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/AMAC_carbon'
assert os.path.isdir(DATA_DIR), f'Expected folder not found: {DATA_DIR} — check the GEE export folder name/path.'

STACK_TIF = os.path.join(DATA_DIR, 'AMAC_S2_HLS6band_stack.tif')
POINTS_CSV = os.path.join(DATA_DIR, 'AMAC_GEDI_points_lonlat_agbd.csv')

for f in [STACK_TIF, POINTS_CSV]:
    print(f, '->', 'FOUND' if os.path.exists(f) else 'MISSING — check the export finished and the filename matches')

## 3. Load the raster stack and GEDI points

The GeoTIFF is the whole-AMAC 6-band composite; the CSV has one row per GEDI footprint (lon, lat, agbd). We'll extract a small image patch around each point next.

In [ ]:
import rasterio
import pandas as pd
import numpy as np

points = pd.read_csv(POINTS_CSV)
print('GEDI points loaded:', len(points))
print(points.head())

src = rasterio.open(STACK_TIF)
print('Raster shape (bands, height, width):', src.count, src.height, src.width)
print('Raster CRS:', src.crs)
print('Raster band descriptions:', src.descriptions)

## 4. Patch extraction

Prithvi-EO-2.0 was pretrained on 224×224 HLS chips (patch size 14×14 for the 300M/600M configs). Following the precedent in the Prithvi-EO-2.0 paper for small-footprint point data (their Sen4Map experiments upsampled small patches to 224×224 rather than retraining the patch-embedding layer), we extract a small native-resolution window around each GEDI point and upsample it to 224×224.

In [ ]:
PATCH_PX = 16          # native 10 m pixels extracted around each point (16x16 = 160m x 160m window)
TARGET_SIZE = 224      # resized to match Prithvi-EO-2.0's pretrained input size

import torch
import torch.nn.functional as F

def extract_patch(src, lon, lat, patch_px=PATCH_PX):
    """Extract a patch_px x patch_px window (all bands) centered on (lon, lat).
    Returns None if the point is too close to the raster edge for a full window."""
    row, col = src.index(lon, lat)
    half = patch_px // 2
    row0, row1 = row - half, row + half
    col0, col1 = col - half, col + half
    if row0 < 0 or col0 < 0 or row1 >= src.height or col1 >= src.width:
        return None
    window = rasterio.windows.Window(col0, row0, patch_px, patch_px)
    patch = src.read(window=window)  # shape: (bands, patch_px, patch_px)
    if np.isnan(patch).any() or (patch == src.nodata).any():
        return None
    return patch.astype(np.float32)

# Quick sanity check on the first few points before running the full extraction
test_patch = None
for i in range(5):
    test_patch = extract_patch(src, points.lon.iloc[i], points.lat.iloc[i])
    print(f'Point {i}: patch =', 'OK ' + str(test_patch.shape) if test_patch is not None else 'DROPPED (edge/nodata)')

## 5. Build the full patch dataset

This extracts a patch for every GEDI point, drops points too close to the raster edge, and resizes each patch to 224×224 via bilinear interpolation. Expect to lose a small fraction of points to the edge check — print the retention rate so it's visible.

In [ ]:
patches, labels, lons, lats = [], [], [], []

for _, row in points.iterrows():
    patch = extract_patch(src, row.lon, row.lat)
    if patch is None:
        continue
    patches.append(patch)
    labels.append(row.agbd)
    lons.append(row.lon)
    lats.append(row.lat)

print(f'Retained {len(patches)} / {len(points)} points ({100*len(patches)/len(points):.1f}%) after edge/nodata filtering.')

patches = np.stack(patches)   # (N, 6, PATCH_PX, PATCH_PX)
labels = np.array(labels, dtype=np.float32)
lons = np.array(lons)
lats = np.array(lats)
print('Patch array shape:', patches.shape)

## 6. Spatially blocked train/val split

The manuscript's Section 4.6 (and the Section 6 caveats) flag random splitting as a leakage risk in a compact AOI. This time we do it properly: split AMAC into a coarse spatial grid and assign whole grid cells to train or validation, so nearby points can't leak across the split.

In [ ]:
N_GRID = 6  # 6x6 spatial grid over AMAC's bounding box; increase for finer blocking if you have more points

lon_bins = np.linspace(lons.min(), lons.max(), N_GRID + 1)
lat_bins = np.linspace(lats.min(), lats.max(), N_GRID + 1)
cell_id = (np.digitize(lons, lon_bins) * 100 + np.digitize(lats, lat_bins))

rng = np.random.default_rng(42)
unique_cells = np.unique(cell_id)
rng.shuffle(unique_cells)
n_val_cells = max(1, int(0.3 * len(unique_cells)))
val_cells = set(unique_cells[:n_val_cells])

is_val = np.array([c in val_cells for c in cell_id])
train_idx = np.where(~is_val)[0]
val_idx = np.where(is_val)[0]

print(f'Spatially blocked split: {len(train_idx)} train / {len(val_idx)} val '
      f'({len(unique_cells)} grid cells total, {n_val_cells} held out for validation)')

## 7. PyTorch Dataset

Normalizes each patch to Prithvi's expected reflectance scale and resizes to 224×224.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class AMACPatchDataset(Dataset):
    def __init__(self, patches, labels, indices, target_size=TARGET_SIZE):
        self.patches = patches[indices]
        self.labels = labels[indices]
        self.target_size = target_size

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.patches[idx])  # (6, H, W), already reflectance (0-1ish) from GEE S2_SR/10000
        x = x.unsqueeze(0)  # add batch dim for interpolate
        x = F.interpolate(x, size=(self.target_size, self.target_size), mode='bilinear', align_corners=False)
        x = x.squeeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

train_ds = AMACPatchDataset(patches, labels, train_idx)
val_ds = AMACPatchDataset(patches, labels, val_idx)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))

## 8. Load the Prithvi-EO-2.0 backbone via TerraTorch — ⚠️ FRAGILE

TerraTorch's backbone registry names have shifted between versions. **Run the discovery line first** and check that a Prithvi entry actually appears before trying to build it — don't assume the exact string below is still correct.

In [ ]:
from terratorch.registry import BACKBONE_REGISTRY

# Discovery step: list what's actually available in your installed terratorch version.
available = [b for b in BACKBONE_REGISTRY if 'prithvi' in b.lower()]
print('Available Prithvi backbones in this TerraTorch install:')
for b in available:
    print(' -', b)
print()
print('If the name used below is not in this list, replace it with one that is.')

In [ ]:
# Common name as of TerraTorch's late-2025 releases; VERIFY against the printout above.
BACKBONE_NAME = 'prithvi_eo_v2_300'

backbone = BACKBONE_REGISTRY.build(
    BACKBONE_NAME,
    pretrained=True,
    num_frames=1,          # single timestamp (dry-season composite), not a time series
    bands=['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2'],
)
print(backbone.__class__.__name__, 'loaded.')

# Sanity check: run one batch through the backbone to confirm the output shape/embedding dim
# before building the regression head around it.
sample_x, _ = next(iter(train_loader))
with torch.no_grad():
    out = backbone(sample_x)
print('Backbone output type:', type(out))
print('Backbone output shape (or first element shape if a list/tuple):',
      out.shape if hasattr(out, 'shape') else [o.shape for o in out])

## 9. LoRA fine-tuning + regression head

Following the label-efficiency precedent from Prithvi-EO-2.0's own AGB fine-tuning (LoRA retained most full-fine-tuning accuracy at a fraction of the trainable parameters), we freeze the backbone and add LoRA adapters to its attention layers, plus a small MLP regression head on the pooled output embedding.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['qkv', 'proj'],  # ⚠️ VERIFY: exact attention layer names depend on the backbone's
                                      # internal module naming — print(backbone) to confirm these exist
                                      # before training; adjust if get_peft_model errors on this list.
    lora_dropout=0.1,
    bias='none',
)

backbone_lora = get_peft_model(backbone, lora_config)
backbone_lora.print_trainable_parameters()

embed_dim = out.shape[-1] if hasattr(out, 'shape') else out[0].shape[-1]

class AGBRegressor(torch.nn.Module):
    def __init__(self, backbone, embed_dim):
        super().__init__()
        self.backbone = backbone
        self.head = torch.nn.Sequential(
            torch.nn.LayerNorm(embed_dim),
            torch.nn.Linear(embed_dim, 128),
            torch.nn.GELU(),
            torch.nn.Linear(128, 1),
        )

    def forward(self, x):
        feats = self.backbone(x)
        if not hasattr(feats, 'shape'):
            feats = feats[-1]  # take the last hidden state if the backbone returns a list of stages
        if feats.dim() == 3:
            feats = feats.mean(dim=1)  # mean-pool over patch tokens -> (batch, embed_dim)
        return self.head(feats).squeeze(-1)

model = AGBRegressor(backbone_lora, embed_dim)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print('Model on device:', device)

## 10. Training loop

In [ ]:
import torch.optim as optim

EPOCHS = 15
LR = 1e-4

optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
loss_fn = torch.nn.MSELoss()

history = {'train_loss': [], 'val_rmse': []}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    train_loss = total_loss / len(train_ds)

    model.eval()
    sq_errs = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            pred = model(x).cpu().numpy()
            sq_errs.extend(((pred - y.numpy()) ** 2).tolist())
    val_rmse = float(np.sqrt(np.mean(sq_errs)))

    history['train_loss'].append(train_loss)
    history['val_rmse'].append(val_rmse)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train MSE loss: {train_loss:.3f} | val RMSE: {val_rmse:.3f} Mg/ha')

## 11. Final evaluation — compare against the Section 6 RF baseline

RF baseline from the manuscript (Section 6, Table 4): **RMSE = 69.72 Mg/ha, MAE = 18.11 Mg/ha, R² = 0.453** (random split). This GFM evaluation uses the spatially blocked split instead, so treat the comparison as directional, not a strictly apples-to-apples number — note that explicitly if you report both in the manuscript.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        pred = model(x).cpu().numpy()
        all_preds.extend(pred.tolist())
        all_true.extend(y.numpy().tolist())

all_preds = np.array(all_preds)
all_true = np.array(all_true)

rmse_final = float(np.sqrt(np.mean((all_preds - all_true) ** 2)))
mae_final = mean_absolute_error(all_true, all_preds)
r2_final = r2_score(all_true, all_preds)

print('=== GFM (Prithvi-EO-2.0 + LoRA) — spatially blocked validation ===')
print(f'RMSE: {rmse_final:.2f} Mg/ha')
print(f'MAE:  {mae_final:.2f} Mg/ha')
print(f'R²:   {r2_final:.3f}')
print()
print('=== RF baseline (manuscript Section 6, random split) ===')
print('RMSE: 69.72 Mg/ha | MAE: 18.11 Mg/ha | R²: 0.453')

## 12. Save results

Saves metrics to a CSV in Drive so they can be pulled back into the manuscript, and saves the LoRA adapter weights (small, a few MB) rather than the full backbone.

In [ ]:
import json

results = {
    'model': 'Prithvi-EO-2.0 + LoRA',
    'n_train': len(train_idx),
    'n_val': len(val_idx),
    'split_type': 'spatially_blocked_grid',
    'rmse_mgha': rmse_final,
    'mae_mgha': mae_final,
    'r2': r2_final,
    'epochs': EPOCHS,
}

out_path = os.path.join(DATA_DIR, 'AMAC_GFM_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print('Saved results to', out_path)

backbone_lora.save_pretrained(os.path.join(DATA_DIR, 'AMAC_prithvi_lora_adapter'))
print('Saved LoRA adapter weights.')

## Next steps

1. If any ⚠️ FRAGILE cell errors out, paste the exact error back and we'll debug it the same way we worked through the GEE script — one concrete error at a time.
2. Once this runs cleanly, send me the printed RMSE/MAE/R² and I'll update the manuscript's Section 7 with a real head-to-head comparison against the Section 6 RF baseline.
3. Longer term: replace the GEDI-only labels here with your field-inventory AGB values (Section 4.2/4.4) once digitized, to break the GEDI-as-label-and-reference circularity noted in Section 6's caveats.